# improved_v3: Phase 4 trên baseline mạnh nhất Phase 3 + T1

Baseline của notebook này là **Phase 3 GLiNER checkpoint (`gliner_phase3_balanced_dev30/final`) + Variant T1**: thresholds/selector/linking kế thừa từ `configs/improved_v2.yaml`, nhưng `ner.model` được override sang checkpoint Phase 3. Đây là baseline mạnh nhất hiện tại trước Phase 4 (official best đã ghi nhận: **31.9244**).

Phase 4 A1 chỉ thêm assertion-lite `isNegated` rất bảo thủ bằng ConText proposer + hai Qwen verifier next-token logits; baseline v2 được giữ nguyên.

Notebook chạy trọn trên một **Colab T4 (16 GB)**: hai teacher nạp ở **4-bit**, compute dtype đặt `float16` vì T4 không hỗ trợ bfloat16 hiệu quả. Kiến trúc chi tiết ở `docs/02_method.md`.

## 1. Kiểm tra runtime

Runtime, Change runtime type, chọn **T4 GPU**.

In [1]:
!nvidia-smi

Mon Aug  3 17:21:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone và cài đặt

Colab đã có torch bản CUDA. `pyproject.toml` là nguồn phụ thuộc duy nhất. Cài thêm `.[quant]` để nạp hai teacher ở 4-bit.

In [2]:
!git clone https://github.com/AIVIETNAM-AIO-DinhBao/ViClinicalIE_2 medextract || true
%cd medextract
!pip install -e ".[quant]"    # bitsandbytes cho chế độ 4-bit

# Colab/IPython đôi khi giữ /content trên sys.path, khiến thư mục repo /content/medextract
# bị import như namespace package và che mất package src/medextract đã install editable.
# Đưa src lên đầu sys.path để các cell Python import đúng medextract.config/pipeline.
import sys
from pathlib import Path

REPO_SRC = Path.cwd() / "src"
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

import medextract
import medextract.config

print("Python kernel medextract:", medextract.__file__)
print("Python kernel src:", REPO_SRC)

Cloning into 'medextract'...
remote: Enumerating objects: 342, done.
remote: Counting objects: 100% (342/342), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 342 (delta 83), reused 328 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (342/342), 8.32 MiB | 16.54 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/medextract
Obtaining file:///content/medextract
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 95.5 MB/s eta 

## 3. Self-check

Không cần GPU, không cần knowledge base. Phải in PASS cho cả bốn mục CONFIG / IMPORTS / SCHEMA / PATHS.

In [3]:
!python scripts/selfcheck.py

PASS  CONFIG  (4 configs load, no orphan keys)
PASS  IMPORTS  (35 modules import)
PASS  SCHEMA  (valid accepted; offset/type/candidates violations rejected)
PASS  PATHS  (doc/notebook paths resolve)


## 4. Knowledge base cho bước linking

`improved_v2` chỉ dùng exact-alias lookup trên hai bảng parquet, **không** cần SapBERT và **không** cần FAISS index, nên chỉ cần hai lệnh build dưới đây. Danh mục ICD-10 tiếng Việt (TT06) **đã đi kèm repo** tại `data/kb/raw/`, nên bạn chỉ cần tải RxNorm và đặt vào `data/kb/raw/RXNCONSO.RRF` (xem `INSTALL.md` cho các nguồn RxNorm).

In [4]:
from pathlib import Path

raw_dir = Path("data/kb/raw")
icd_file = raw_dir / "Phu_luc_Bang_danh_muc_ICD10_FINAL_TT06_2026.xlsx"
rxnorm_file = raw_dir / "RXNCONSO.RRF"
print("KB raw files:")
!ls -lh data/kb/raw
if not icd_file.exists():
    raise FileNotFoundError(f"Missing ICD-10 TT06 file from repo: {icd_file}")
if not rxnorm_file.exists():
    raise FileNotFoundError(
        "Missing RXNCONSO.RRF. This file is license-gated, so it is not committed to the public repo. "
        "Upload it to data/kb/raw/RXNCONSO.RRF or copy it from Drive before running build_rxnorm."
    )
print("OK: required KB raw files are available.")


KB raw files:
total 32M
-rw-r--r-- 1 root root 2.9M Aug  3 17:21 Phu_luc_Bang_danh_muc_ICD10_FINAL_TT06_2026.xlsx
-rw-r--r-- 1 root root  30M Aug  3 17:21 RXNCONSO.RRF
OK: required KB raw files are available.


In [5]:
!python -m medextract.kb.build_icd    --tt06    # -> data/kb/processed/icd_terms_v2.parquet
!python -m medextract.kb.build_rxnorm --v2      # -> data/kb/processed/rxnorm_terms_v2.parquet

icd_terms_v2 (TT06): 15,926 aliases / 15,845 codes -> data/kb/processed/icd_terms_v2.parquet
icd_terms_v2: 15,926 rows, 15,845 codes
  I10: ['Bệnh tăng huyết áp vô căn (nguyên phát)']
  E11.9: ['Bệnh đái tháo đường típ 2, không kèm biến chứng']
  J18.9: ['Viêm phổi, không xác định']
  K21.0: ['Bệnh trào ngược dạ dày - thực quản kèm viêm thực quản']
  A00: ['Bệnh tả']
  U07.1: ['COVID-19, virus được xác định', 'COVID-19, vi rút được xác định']
rxnorm_terms_v2: 82,582 aliases / 38,993 rxcui -> data/kb/processed/rxnorm_terms_v2.parquet
tty
SCD     32872
SBD     28690
SCDC    10967
IN       6832
PIN      2121
MIN      1100
rxnorm_terms_v2: 82,582 rows, 38,993 rxcui


## 5. Chọn checkpoint NER Phase 3

Mount Google Drive và tìm checkpoint GLiNER fine-tuned Phase 3. Biến `PHASE3_NER_MODEL` sẽ được dùng để override `ner.model` trong config chạy submission.

Checkpoint đúng cho baseline mạnh nhất: `models/gliner_phase3_balanced_dev30/final`. Cell này **không fallback xuống Phase 1** để tránh vô tình chạy cấu hình yếu hơn.

In [6]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running inside Google Colab; assuming Drive/local paths are already available.")

PHASE3_NER_CANDIDATES = [
    # Output of Phase 3 external NER augmentation notebook.
    Path("/content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final"),
    Path("/content/drive/MyDrive/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final"),

    # Optional local copy inside the cloned repo/session.
    Path("models/gliner_phase3_balanced_dev30/final"),
]

# Optional manual override before running this cell:
# PHASE3_NER_MODEL_OVERRIDE = "/content/drive/.../models/gliner_phase3_balanced_dev30/final"
if "PHASE3_NER_MODEL_OVERRIDE" in globals():
    PHASE3_NER_CANDIDATES.insert(0, Path(PHASE3_NER_MODEL_OVERRIDE))

def is_gliner_model_dir(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "gliner_config.json").exists()
        and ((path / "pytorch_model.bin").exists() or (path / "model.safetensors").exists())
    )

print("Phase 3 NER checkpoint candidates (no Phase 1 fallback):")
for candidate in PHASE3_NER_CANDIDATES:
    status = "OK" if is_gliner_model_dir(candidate) else "missing/incomplete"
    print(f"- {candidate}: {status}")

PHASE3_NER_MODEL = next((p for p in PHASE3_NER_CANDIDATES if is_gliner_model_dir(p)), None)
if PHASE3_NER_MODEL is None:
    raise FileNotFoundError(
        "Could not find GLiNER Phase 3 checkpoint. "
        "Run the Phase 3 external NER augmentation notebook first, copy its final folder to "
        "Drive: Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final, "
        "or set PHASE3_NER_MODEL_OVERRIDE before running this cell."
    )

PHASE3_NER_MODEL = str(PHASE3_NER_MODEL)
print("Using Phase 3 NER model:", PHASE3_NER_MODEL)

Mounted at /content/drive
Phase 3 NER checkpoint candidates (no Phase 1 fallback):
- /content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final: OK
- /content/drive/MyDrive/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final: missing/incomplete
- models/gliner_phase3_balanced_dev30/final: missing/incomplete
Using Phase 3 NER model: /content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final


## 6. Config Colab T4: Phase 3 checkpoint + Variant T1

Cell này tạo `colab_t4_t1.yaml`, tức baseline mạnh nhất: kế thừa `configs/improved_v2.yaml` cho Variant T1 (per-type thresholds, raw_floor thấp, Qwen consensus selector, exact-alias linking), rồi override `ner.model` sang checkpoint Phase 3.

Baseline này chưa bật assertions; cell Phase 4 bên dưới sẽ tạo config riêng extend từ `colab_t4_t1.yaml`.

In [7]:
import pathlib

# Repo id của hai teacher trên Hugging Face Hub.
# Nếu bạn đã tải sẵn trọng số về máy, thay bằng đường dẫn cục bộ.
PRIMARY_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
SECONDARY_MODEL = "Qwen/Qwen3.5-4B"

if "PHASE3_NER_MODEL" not in globals():
    raise RuntimeError("Run the Phase 3 checkpoint cell before creating colab_t4_t1.yaml")

override = f"""# Colab T4 config: Phase 3 GLiNER checkpoint + Variant T1.
# Baseline mạnh nhất trước Phase 4: official best 31.9244.
# T1 kế thừa từ configs/improved_v2.yaml: per-type thresholds, raw_floor thấp,
# Qwen consensus correction/additions, exact-alias linking, empty assertions.
extends: configs/improved_v2.yaml
# IMPORTANT: keep this exactly improved_v2 so run.py/build_pipeline uses PipelineV2.
solution: improved_v2

ner:
  model: "{PHASE3_NER_MODEL}"
  max_chunk_chars: 800
  raw_floor: 0.02
  label_map:
    "TRIỆU_CHỨNG": "TRIỆU_CHỨNG"
    "CHẨN_ĐOÁN": "CHẨN_ĐOÁN"
    "THUỐC": "THUỐC"
    "TÊN_XÉT_NGHIỆM": "TÊN_XÉT_NGHIỆM"
    "KẾT_QUẢ_XÉT_NGHIỆM": "KẾT_QUẢ_XÉT_NGHIỆM"
  thresholds:
    TRIỆU_CHỨNG: 0.20
    CHẨN_ĐOÁN: 0.25
    THUỐC: 0.30
    TÊN_XÉT_NGHIỆM: 0.15
    KẾT_QUẢ_XÉT_NGHIỆM: 0.35

consensus_selector:
  enabled: true
  primary_model: {PRIMARY_MODEL}
  secondary_model: {SECONDARY_MODEL}
  primary_device: cuda:0
  secondary_device: cuda:0
  batch_size: 16
  max_length: 384
  addition_margin_none: 1.0
  addition_types: [TRIỆU_CHỨNG, TÊN_XÉT_NGHIỆM, KẾT_QUẢ_XÉT_NGHIỆM]

quantization:
  mode: 4bit
  dtype: float16
  compute_dtype: float16
  double_quant: true
"""

pathlib.Path("colab_t4_t1.yaml").write_text(override, encoding="utf-8")
print(override)

# Colab T4 config: Phase 3 GLiNER checkpoint + Variant T1.
# Baseline mạnh nhất trước Phase 4: official best 31.9244.
# T1 kế thừa từ configs/improved_v2.yaml: per-type thresholds, raw_floor thấp,
# Qwen consensus correction/additions, exact-alias linking, empty assertions.
extends: configs/improved_v2.yaml
# IMPORTANT: keep this exactly improved_v2 so run.py/build_pipeline uses PipelineV2.
solution: improved_v2

ner:
  model: "/content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final"
  max_chunk_chars: 800
  raw_floor: 0.02
  label_map:
    "TRIỆU_CHỨNG": "TRIỆU_CHỨNG"
    "CHẨN_ĐOÁN": "CHẨN_ĐOÁN"
    "THUỐC": "THUỐC"
    "TÊN_XÉT_NGHIỆM": "TÊN_XÉT_NGHIỆM"
    "KẾT_QUẢ_XÉT_NGHIỆM": "KẾT_QUẢ_XÉT_NGHIỆM"
  thresholds:
    TRIỆU_CHỨNG: 0.20
    CHẨN_ĐOÁN: 0.25
    THUỐC: 0.30
    TÊN_XÉT_NGHIỆM: 0.15
    KẾT_QUẢ_XÉT_NGHIỆM: 0.35

consensus_selector:
  enabled: true
  primary_model: Qwen/Qwen3-4B-Instruct-2507
  secondary_model: Qwen/Qwen3.5-4B
  primary_

## 6b. Verify config dispatch

Cell này không chạy inference; nó chỉ load YAML và kiểm tra `solution` vẫn là `improved_v2`, selector Qwen đang bật, và Phase 4 config kế thừa đúng baseline T1. Nếu cell này fail thì **đừng chạy submission**.

In [8]:
import sys
from pathlib import Path

REPO_SRC = Path.cwd() / "src"
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

from medextract.config import load_config

base_cfg = load_config("colab_t4_t1.yaml")
print("solution:", base_cfg.get("solution"))
print("NER model:", base_cfg.get("ner", {}).get("model"))
print("selector enabled:", base_cfg.get("consensus_selector", {}).get("enabled"))
print("primary:", base_cfg.get("consensus_selector", {}).get("primary_model"))
print("secondary:", base_cfg.get("consensus_selector", {}).get("secondary_model"))
assert base_cfg.get("solution") == "improved_v2", "Must be improved_v2 to use PipelineV2/Qwen selector"
assert "gliner_phase3_balanced_dev30/final" in base_cfg.get("ner", {}).get("model", "")
assert base_cfg.get("consensus_selector", {}).get("enabled") is True
print("OK: colab_t4_t1.yaml will dispatch to PipelineV2 and load Qwen selector during run.py")

solution: improved_v2
NER model: /content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final
selector enabled: True
primary: Qwen/Qwen3-4B-Instruct-2507
secondary: Qwen/Qwen3.5-4B
OK: colab_t4_t1.yaml will dispatch to PipelineV2 and load Qwen selector during run.py


## 7. Chạy baseline Phase 3 + T1 và đóng gói bản nộp

Notebook dùng trực tiếp 100 file `.txt` đã có trong `data/input/`. `--zip` ghi `out/improved_v2_t1/submission.zip`, các file JSON nằm phẳng, không có thư mục con.

In [9]:
from pathlib import Path

input_dir = Path("data/input")
txt_files = sorted(input_dir.glob("*.txt"), key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem)
print(f"Found {len(txt_files)} input .txt files in {input_dir}")
print([p.name for p in txt_files[:10]])
if len(txt_files) != 100:
    raise RuntimeError(f"Expected 100 input files in {input_dir}, found {len(txt_files)}")

!rm -rf out/improved_v2_t1
!python run.py --config colab_t4_t1.yaml --input data/input --output out/improved_v2_t1 --zip

!echo "JSON outputs:"
!find out/improved_v2_t1 -maxdepth 1 -type f -name "*.json" | wc -l
!ls -lh out/improved_v2_t1/submission.zip

Found 100 input .txt files in data/input
['1.txt', '2.txt', '3.txt', '4.txt', '5.txt', '6.txt', '7.txt', '8.txt', '9.txt', '10.txt']
2026-08-03 17:22:33,416 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-08-03 17:22:36,355 INFO datasets: TensorFlow version 2.20.0 available.
2026-08-03 17:22:36,356 INFO datasets: JAX version 0.7.2 available.
2026-08-03 17:22:37,211 INFO medextract.ner.gliner: loading GLiNER /content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final on cuda:0
2026-08-03 17:22:37,747 INFO gliner.model: Loading the following GLiNER type: <class 'gliner.model.UniEncoderSpanGLiNER'>...
2026-08-03 17:23:03,089 INFO medextract.models.registry: [registry] + final                        ner          0.289B (cuda:0)
2026-08-03 17:23:03,319 INFO medextract.kb_terms: [ICD10] 15926 aliases / 15845 codes loaded
2026-08-03 17:23:04,055 INFO medextract.kb_terms: [RXNORM] 82582 aliases / 38993 codes loaded
2026-08-03 17:23:04,198 INFO httpx: HT

## 7b. Phase 4 A1: Phase 3 + T1 + Qwen-confirmed negation assertions

Cell này giữ nguyên baseline `colab_t4_t1.yaml`, tạo thêm `colab_t4_t1_assertion_neg.yaml` chỉ bật `isNegated` rất bảo thủ: ConText propose cue phủ định, cả hai Qwen phải xác nhận bằng next-token logits margin >= 2.0. Sau khi chạy, dùng `scripts/audit_assertions.py` để kiểm tra tỷ lệ assertion không rỗng trước khi nộp.

In [10]:
import pathlib
import sys
from pathlib import Path

REPO_SRC = Path.cwd() / "src"
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

phase4 = """# Phase 4 A1: Phase 3 + T1 baseline + conservative Qwen-confirmed negation.
extends: colab_t4_t1.yaml

assertions:
  neg_window_chars: 60
  block_lookback_lines: 6

assertion_selector:
  enabled: true
  labels: [isNegated]
  primary_margin: 2.0
  secondary_margin: 2.0
  context_window_chars: 180
"""
pathlib.Path("colab_t4_t1_assertion_neg.yaml").write_text(phase4, encoding="utf-8")
print(phase4)

from medextract.config import load_config
phase4_cfg = load_config("colab_t4_t1_assertion_neg.yaml")
print("solution:", phase4_cfg.get("solution"))
print("selector enabled:", phase4_cfg.get("consensus_selector", {}).get("enabled"))
print("assertion selector enabled:", phase4_cfg.get("assertion_selector", {}).get("enabled"))
assert phase4_cfg.get("solution") == "improved_v2", "Must be improved_v2 to use PipelineV2/Qwen selector"
assert phase4_cfg.get("consensus_selector", {}).get("enabled") is True
assert phase4_cfg.get("assertion_selector", {}).get("enabled") is True
print("OK: Phase 4 config will dispatch to PipelineV2 with Qwen selector + assertions")

!rm -rf out/improved_v2_t1_assertion_neg
!python run.py --config colab_t4_t1_assertion_neg.yaml --input data/input --output out/improved_v2_t1_assertion_neg --zip

!python scripts/audit_assertions.py --pred out/improved_v2_t1_assertion_neg --input data/input --samples 12
!ls -lh out/improved_v2_t1_assertion_neg/submission.zip

# Phase 4 A1: Phase 3 + T1 baseline + conservative Qwen-confirmed negation.
extends: colab_t4_t1.yaml

assertions:
  neg_window_chars: 60
  block_lookback_lines: 6

assertion_selector:
  enabled: true
  labels: [isNegated]
  primary_margin: 2.0
  secondary_margin: 2.0
  context_window_chars: 180

solution: improved_v2
selector enabled: True
assertion selector enabled: True
OK: Phase 4 config will dispatch to PipelineV2 with Qwen selector + assertions
2026-08-03 17:48:01,044 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-08-03 17:48:02,273 INFO datasets: TensorFlow version 2.20.0 available.
2026-08-03 17:48:02,274 INFO datasets: JAX version 0.7.2 available.
2026-08-03 17:48:02,760 INFO medextract.ner.gliner: loading GLiNER /content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase3_balanced_dev30/final on cuda:0
2026-08-03 17:48:02,768 INFO gliner.model: Loading the following GLiNER type: <class 'gliner.model.UniEncoderSpanGLiNER'>...
2026-08-03 17:48:15,472 INFO me

## 8. Xem một mẫu output

In [11]:
import json, pathlib

# Ưu tiên xem output Phase 4 nếu đã chạy, fallback về baseline Phase 3 + T1.
candidates = [
    pathlib.Path("out/improved_v2_t1_assertion_neg/001.json"),
    pathlib.Path("out/improved_v2_t1/001.json"),
]
p = next((path for path in candidates if path.exists()), candidates[0])
if not p.exists():
    raise FileNotFoundError("Run the baseline or Phase 4 inference cell before viewing sample output.")

data = json.load(open(p, encoding="utf-8"))
print(f"{p}: {len(data)} concept(s)")
print()
print(json.dumps(data[:3], ensure_ascii=False, indent=2))

FileNotFoundError: Run the baseline or Phase 4 inference cell before viewing sample output.

## 9. Chấm điểm local

`score.py` là bản đọc lại công thức của Ban Tổ chức để xếp hạng hai lần chạy local, không phải bộ chấm chính thức. Chuẩn bị thư mục nhãn dạng `<thư mục nhãn>/{stem}.json` cùng schema với bản nộp, rồi:

```bash
python score.py --pred out/improved_v2_t1 --gold <thư mục nhãn> -v
python score.py --pred out/improved_v2_t1_assertion_neg --gold <thư mục nhãn> -v
```